# Minimal surfaces with differential forms

## Introduction

Welcome! 

This blog post is the first in a series of posts discussing and solving the famous Plateau's problem in geometry.

In this introductory post, we will state and define the problem rigorously, discuss on some of the most common approaches to solving it.

## Plateau's Problem

### Definition

Let $M$ be a three-dimensional compact ambient space, assumed to be a bounded subset $M \subset \mathbb{R}^3$ with the Euclidean metric. The classical Plateau problem is formulated as follows:

> Given a closed curve $\Gamma \subset M$, find an oriented surface $\Sigma \subset M$ bordered by $\Gamma$ that minimizes the area functional.

Mathematically, this problem is expressed as:

$$
\min_{\Sigma \, : \, \partial \Sigma = \Gamma} \text{Area}(\Sigma).
$$

### Examples

For example, the circle $\Gamma = \mathbb{S}^1$ has a simple disk solution as its minimal surface.

!TODO: Place photo here!

And for a more complex example, the helix with a central trunk

## Common solution methods

Many different approaches to this problem have been posited, for example

- In 1927 Jesse Douglas created a finite difference method for solving this problem [1].
- Mean curvature flow based approach in 1990 by Dziuk et al.
- $H^1$ Sobolev gradient flow approach in 1993 by Pinkall and Polthier

However, our solution method is based on more recent mathematical theory, more specifically it relies on the machinery of *geometric measure theory* where we model surfaces and curves as **integral currents**.  

We will continue discussion of our solution in the [next part]().

# References

### About this notebook

This notebook is the whole project in one place: the mathematics, the
discretization, the solver, and the validation. It follows

> Stephanie Wang and Albert Chern, *Computing Minimal Surfaces with Differential
> Forms*, ACM TOG 40(4), 2021.

The algorithms live in `src/` and are imported here rather than defined inline,
so the exposition and the tested code cannot drift apart. Several published
details turned out to be wrong; those are collected in the final section and in
`README.md`, each with a test guarding it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from src import curves, extract, spectral
from src.grid import Grid
from src.initial_guess import compute_initial_guess
from src.plateau import solve_plateau

%load_ext autoreload
%autoreload 2

# Curves and surfaces with differential forms

## Introduction

This is the second part of our exploration of the Plateau problem and the approach taken to solve it in the paper by Wang and Chern.

In this part, we will focus on defining the problem using vector fields and differential forms. As the original paper authors state, doing this reformalization allows us to consider the problem as a convex optimization problem, which makes finding the correct solution easier and more stable. Formulating the problem in this matter does require us however to go through a couple of mathematical definitions and examples. We do this in order to obtain a comprehensive overview of the solution method.

We begin by discussion of vector fields on smooth manifolds, then move on to differential forms and then finally restate the Plateau problem using objects known as dirac-delta differential forms.

## Vector fields

The basic object of study is a vector field. Usually vector fields are first introduced as simply functions on some euclidean space (2 or 3 -dimensional) which map an "arrow" to each point in space. Formally, for example a 2-dimensional vector field is a function $f: \mathbb{R}^2 \to \mathbb{R}^2$, usually with some notion of smoothness included as well to make the function behave _nicely_.

The basic definition works well in an Euclidean setting and allows us to define some other operators, such as the divergence and curl operators (in suitable dimensions). Furthermore, it allows a simple mental picture of what is going on. One can image the speed and direction of wind on the globe as a vector field defined on the 2-sphere, as the prototypical example.

However, in more general context of differential geometry and smooth manifolds, this basic definition does not make sense. In the sense that if we don't have global coordinates for our space, but simply a local coordinate chart at each point $p$ on the smooth manifold $M$, then we cannot define this vector field globally for each point on the surface. 

To remedy this, we can modify the definition of a vector field to be more abstract using things called _derivations_.

### Vector fields as derivations

At each point $p \in M$, we consider all the smooth real-valued functions on $M$, denoted $C^\infty(M)$. A tangent vector at $p$ can then be thought of as a **directional derivative operator** acting on these functions. That is, a vector $X_p \in T_pM$ is defined as a linear map
$X_p : C^\infty(M) \to \mathbb{R}$ which satisfies the **Leibniz rule**:

$$
X_p(fg) = X_p(f)g(p) + f(p)X_p(g)
$$

This is exactly the rule you expect from a derivative operator—it’s how the derivative of a product behaves. Such a map is called a **derivation**. A vector field $X$ is then the map from a point $p \in M$ to this linear map $X_p$. Formally, $X: p \mapsto X_p$. And the application of vector field to a smooth function produces a map from a point to a real number, $X(f): p \mapsto \mathbb{R}$.

### Why is this useful?

This approach gives us a definition of vectors that:

- **Works on any smooth manifold**, whether or not we have coordinates.
- Is **intrinsic**, i.e., does not rely on choosing coordinates or embedding the manifold in some higher-dimensional space.
- Naturally leads to the dual concept of differential forms, which are linear functionals on vectors.


### Coordinate Basis Vectors and Derivations

In $\mathbb{R}^n$, we usually express a vector as a linear combination of the standard basis vectors: $v = a^1 e_1 + a^2 e_2 + \dots + a^n e_n$, where each $e_i$ points in the direction of the $x^i$-axis. On a smooth manifold $M$, there is no global coordinate system, but around any point $p \in M$, we can choose a local coordinate chart $(x^1, \dots, x^n)$. In this chart, we define the **coordinate vector fields** $\left( \frac{\partial}{\partial x^1}, \dots, \frac{\partial}{\partial x^n} \right)$ using differentiation: $\frac{\partial}{\partial x^i} (f)(p) = \frac{\partial f}{\partial x^i}(p)$. In other words, a coordinate vector field simply differentiates the input function $f \in C^\infty(M)$ at point $p$ in the "direction" of the specific coordinate, leaving other coordinates alone, exactly the partial derivative by definition.

We define the **tangent space** $T_pM$ to be the set of all derivations at $p$. The collection $\left( \frac{\partial}{\partial x^1} \right)_p, \dots, \left( \frac{\partial}{\partial x^n} \right)_p$ forms a **basis** of $T_pM$. Therefore, any vector $X_p \in T_pM$ can be uniquely expressed as some linear combination $X_p = a^1 \left( \frac{\partial}{\partial x^1} \right)_p + \dots + a^n \left( \frac{\partial}{\partial x^n} \right)_p$ and acts on functions as a directional derivative:
$X_p(f) = a^1 \frac{\partial f}{\partial x^1}(p) + \dots + a^n \frac{\partial f}{\partial x^n}(p)$. So in local coordinates, a vector at a point $p$ is fully described by how it differentiates functions along coordinate directions.

### Example

Let $f(x, y) = x^2 + x y$ and let $p = (1, 2)$. Define a vector field $X_q = 3 y\frac{\partial}{\partial x} - x\frac{\partial}{\partial y}$, for point $q = (x, y)$.  
Therefore the derivation (vector) at $p$, $X_p$ is $3 \cdot 2 \frac{\partial}{\partial x} - \frac{\partial}{\partial y} = 6 \frac{\partial}{\partial x} - \frac{\partial}{\partial y}$.  
We compute the partial derivatives, $\frac{\partial f}{\partial x} = 2x + y$, and $\frac{\partial f}{\partial y} = x$. At the point $p = (1, 2)$, we get $\frac{\partial f}{\partial x}(p) = 4$ and $\frac{\partial f}{\partial y}(p) = 1$.  
Thus, $X_p(f) = 6 \cdot 4 - 1 \cdot 1 = 23$.  

Picking another point $q = (-2, 0)$, we obtain another value for the application of the vector field $X$ to $f$, $X_q(f) = (3 \cdot 0 \frac{\partial}{\partial x} + 2 \frac{\partial}{\partial y})(f)(q) = -4$.

### Summary

Vectors on manifolds are best understood as **derivations** — operators acting on smooth functions, satisfying the Leibniz rule. The coordinate derivations $\frac{\partial}{\partial x^i}$ provide a natural basis for $T_pM$, and every vector can be written as a combination of them. This formalism is local, intrinsic, and independent of any embedding into Euclidean space.

This new view becomes extremely powerful when combined with differential forms, because it provides the foundation for defining integration, exterior differentiation, and the general machinery of geometric analysis on manifolds.

## Differential forms

Differential forms are the next level up from vector fields, a differential form is defined as a linear functional on the space of vector fields. In other words, a differential 1-form takes a vector field $X$ and produces a real number $\mathbb{R}$ for each point on the manifold $M$. Likewise, a differential $k$-form takes $k$ vector fields and produces a real number for every point on the manifold. Also, a differential 0-form is defined to be just a scalar function, that is a function from the manifold $M$ to real numbers. Intuitively, a differential form measures how well the vector field aligns with the form at some point on the manifold. A good way of visualizing differential forms in low dimensions can be found for example [here by Dan Piponi](http://yaroslavvb.com/papers/notes/piponi-on.pdf)(1998). 

### Differential form basis

A differential 1-form can look for example something like this, $\omega = 2 dx - 5xdy$, the quantities $dx$ and $dy$ are the dual 1-forms of the coordinate directions $\frac{\partial}{\partial x}, \frac{\partial}{\partial y}$, they are defined to be such that the application of the basis form to the corresponding coordinate basis vector produces simply $1$ for all points $p \in M$. This hints to the true fact that like vector fields, 1-forms can be made formed by a simple linear combination of those basis 1-forms.
 
Since there can be however differential forms with higher degrees, for example $2$-forms, we can use the wedge product ($\wedge$) to combine lower degree forms to build higher ones. For example we can have a 2-form on 3-dimensional manifold, like $\eta = 2 dx \wedge dy - 7 xdy \wedge dz + xy dx \wedge dz$. The details of the wedge product are left outside of this blog post for brevity, but details can be found on any standard differential geometry book.

### Differential form example

For a motivating example, the application of a 1-form $\omega = 2\,dx - 5x\,dy$.
Taking the point $p = (1, 3)$ and a vector field, $X = y \frac{\partial}{\partial x} + x \frac{\partial}{\partial y}$. At point $p$, the specific vector is $X_p = 3 \frac{\partial}{\partial x} + 1 \frac{\partial}{\partial y}$.  
Then we apply the 1-form $\omega$ to this vector, obtaining $\omega(X_p) = 2 \cdot 3 - 5 \cdot 1 \cdot 1 = 6 - 5 = 1$.  
So at the point $p = (1, 3)$, the 1-form $\omega = 2 dx - 5x dy$ evaluates to $1$ when applied to the vector field $X$.

If we drop the specific point $p$ but rather look at the 1-form 

This gives an idea of how 1-forms act like "detectors" or "probes" that measure the components of vector fields, weighted by their coefficients and position on the manifold.

### Differential 2-form example

Let us now consider a differential 2-form, which takes in two vectors and produces a real number. A typical example in $\mathbb{R}^3$ might look like:$\eta = x\,dy \wedge dz + y\,dz \wedge dx + z\,dx \wedge dy$

Evaluated on a pair of vectors $v = \frac{\partial}{\partial y} + \frac{\partial}{\partial z}, w = \frac{\partial}{\partial x} + 2\frac{\partial}{\partial z}$, the value of the 2-form at point $p = (1, 2, 3)$ is computed using the wedge products.

$dy \wedge dz (v, w)$:
$$
dy \wedge dz (v, w) = \det \begin{bmatrix} v^y & v^z \\ w^y & w^z \end{bmatrix} = \det \begin{bmatrix} 1 & 1 \\ 0 & 2 \end{bmatrix} = 2
$$

$dz \wedge dx (v, w)$:
$$
\det \begin{bmatrix} v^z & v^x \\ w^z & w^x \end{bmatrix} = \det \begin{bmatrix} 1 & 0 \\ 2 & 1 \end{bmatrix} = 1
$$

$dx \wedge dy (v, w)$:
$$
\det \begin{bmatrix} v^x & v^y \\ w^x & w^y \end{bmatrix} = \det \begin{bmatrix} 0 & 1 \\ 1 & 0 \end{bmatrix} = -1
$$

Now we evaluate $\eta(v, w)$ at $p = (1, 2, 3)$:
- The first term gives $x \cdot dy \wedge dz (v, w) = 1 \cdot 2 = 2$
- The second term gives $y \cdot dz \wedge dx (v, w) = 2 \cdot 1 = 2$
- The third term gives $z \cdot dx \wedge dy (v, w) = 3 \cdot (-1) = -3$

Putting it together:

$$
\eta(v, w) = 2 + 2 - 3 = 1
$$

So the 2-form $\eta$ evaluates to 1 on the vectors $v$ and $w$ at the point $(1, 2, 3)$.




## Dirac-delta forms and restating Plateau's problem

Having introduced differential forms and their role as intrinsic, coordinate-free tools for probing vector fields and oriented geometry, we are now ready to discuss a class of generalized forms that plays a central role in reformulating the Plateau problem: **Dirac-delta differential forms**. These are not smooth forms in the usual sense, but rather distributions—or more precisely, **currents**, which are continuous linear functionals on the space of smooth compactly supported differential forms. They allow us to represent highly singular geometric objects, such as curves or surfaces, as objects that can still be integrated against smooth test forms.

The key idea is to represent a surface $S \subset M$ not by an explicit parametrization or an embedding, but by the **1-form current** $\delta_S$, which satisfies the identity
$$
\int_M \omega \wedge \delta_S = \int_S \omega
$$
for any smooth 2-form $\omega$ on the ambient manifold $M$. This object behaves like a "generalized differential form" that is zero almost everywhere, but supported entirely on the surface $S$. It captures the geometry of $S$ in the weak sense—via its action on test forms—making it ideal for variational formulations where smoothness may not be guaranteed.

Intuitively, we can visualize dirac-delta forms $\delta$ representing curves and surfaces as "impulses". As normal differential forms can be visualized as vector fields using [musical isomorphisms]() in 3-dimensions, these dirac-delta forms can be visualized in a similar way as being connected to vector fields but somehow being more "local". This locality can be realized as the vector field vanishing as we stray too far from the associated curve or surface.


# Visualization of dirac-delta forms

Here we have a couple of different visualization of dirac-delta differential forms, as stated previously, the dirac-delta forms are differential forms in the weak sense that they are localized to a specific submanifold. In our setting of 3-dimensional euclidean space, the ambient space $M = \mathbb{R}^3$ and therefore a curve $\Gamma$ can be realized using a dirac delta $2$-form. Similarly, a 2-dimensional surface $S$ can be realized as a dirac-delta $1$-form.

## Representation of a curve

Below we build the Dirac-δ 2-form of a circle directly and look at it. The field
`delta_gamma` is the discrete current density of the curve: its flux through any
surface is that surface's signed intersection number with Γ.

In [ ]:
grid = Grid(48)
radius = 0.3
gamma = lambda t: curves.circle(t, radius=radius)

guess = compute_initial_guess(grid, gamma)

print(f"d(eta_0) = delta_Gamma  to relative {guess.residual:.2e}")
print(f"area vector A = {np.round(guess.area, 5)}   (exact: [0, 0, {np.pi*radius**2:.5f}])")
print(f"mean(eta_0)   = {np.round(guess.eta_0.reshape(-1, 3).mean(axis=0), 5)}")

In [ ]:
mid = grid.resolution // 2
X, Y = grid.positions_grid[:, :, mid, 0], grid.positions_grid[:, :, mid, 1]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
m = spectral.pointwise_norm(guess.delta_gamma)[:, :, mid]
axes[0].contourf(X, Y, m, levels=60, cmap="inferno")
axes[0].set_title(r"$|\delta_\Gamma|$ in the plane of the curve")
axes[1].contourf(X, Y, guess.eta_0[:, :, mid, 2], levels=60, cmap="viridis")
axes[1].set_title(r"$z$-component of $\eta_0$")
for ax in axes:
    ax.add_artist(plt.Circle((0.5, 0.5), radius, fill=False, color="w", ls="--"))
    ax.set_aspect("equal")
plt.tight_layout()

In the above plot, we have a visualization of a dirac-delta $2$-form, representing a curve $\Gamma$. The curve is displayed as a black solid line, while the dirac-delta curve representation is displayed as blue streamlines.

The visualization of the dirac-delta form is based on the musical isomorphism correspondence with the curve tangent vector field. More details on how this visualization is made can be found in the original paper.  
This mental image is very helpful to keep in mind in the next sections as we dive into the actual implementation, the key idea being that we want to __represent a geometric object as a localized differential form__.

The manim cell below renders the same object as animated streamlines. It needs
the optional animation extra (`pip install -e ".[animation]"`).

In [ ]:
# %load_ext manim
# from src.animation.animate_field import build_vector_field_scene
#
# %%manim -ql -v WARNING -o delta_gamma.mp4 scene
# scene = build_vector_field_scene(grid, guess.delta_gamma, curve_function=gamma,
#                                  streak_scaling=0.02)

# From Geometry to Convexity: Formulating the Minimal Surface Optimization Problem

> Based on the work of Stephanie Wang and Albert Chern,  
> [*Computing Minimal Surfaces with Differential Forms*](https://doi.org/10.1145/3450626.3459781),  
> ACM Transactions on Graphics, 2021

In the previous chapter, we introduced the mathematical machinery of Dirac-δ forms, differential forms, and vector fields in differential geometry. Now, we’ll put that machinery to work and formulate the Plateau problem as a convex optimization problem —the foundation of the minimal surface algorithm developed by Wang and Chern.

As the authors state, this convex formulation bypasses the need for meshes or parametric surface representations. It works directly in the ambient space, supports topological flexibility, and is robust under numerical discretization. Most importantly, it guarantees convergence to the global minimum.

## 1. The Classical Plateau Problem

The original problem statement was "given a closed curve $\Gamma \subset \mathbb{R}^3$, find a surface $\Sigma$ with boundary $\partial \Sigma = \Gamma$ that minimizes the area". In other words, find the "soap film surface" for a given wireframe. Mathematically, the problem can be stated as
$$
\min_{\Sigma : \partial \Sigma = \Gamma} \text{Area}(\Sigma)
$$

However, as the authors state, this version of them problem is difficult because:
- The space of surfaces is non-convex, i.e. no clear direction for minimum surface area
- Topology matters: different surfaces with the same boundary can have different numbers of holes
- Mesh-based methods are unstable if the topology is incorrect

These difficulties can however be overcome if we formalize the surface not as a strict geometric object, but rather as a dirac-delta distribution. Formally, we represent a surface $\Sigma \subset \mathbb{R}^3$ by a 1-form $\delta_\Sigma \in \Omega^2(M)$. Similarly, we can represent the boundary curve $\Gamma$ by a 2-form $\delta_\Gamma \in \Omega^1(M)$. As a reminder, $\Omega^k(M)$ denotes the set of smooth differential $k$-forms on the manifold $M$.

The dirac-delta forms satisfy these integral properties,
$$
\int_M \omega \wedge \delta_\Sigma = \int_\Sigma \omega
$$
$$
\int_M \eta \wedge \delta_\Gamma = \int_\Gamma \eta
$$
Therefore, the dirac-delta forms $\delta_\Gamma$ and $\delta_\Sigma$ characterize the geometric objects in a weak sense. Similar to weak solutions of partial differential equations.  
Using these forms, the original authors reformulate the problem by defining a mass norm for a 1-form $\eta$, $\| \eta \|_{\text{mass}} = \sup_{\omega \in \Omega^2(M), \|\omega\|_{\max} \leq 1} \int_M \omega \wedge \eta$. Then, the original problem is just to find the surface representation form with minimum $\| \delta_\Sigma \|_{\text{mass}}$

## 2. Rewriting the Boundary Condition

For enforcing the boundary condition $\partial \Sigma = \Gamma$, the authors derive the following weak formulation using a "test" 1-form $\eta$ and Stoke's theorem, stating that for any such test form must hold 

$$
\int_\Gamma \eta = \int_\Sigma d\eta = \int_M d\eta \wedge \delta_\Sigma = \int_M \eta \wedge d\delta_\Sigma
\Rightarrow d\delta_\Sigma = \delta_\Gamma
$$

So the boundary constraint becomes simply $d \delta_\Sigma = \delta_\Gamma$, which is a constraint based on the exterior derivative operator $d$. Most importantly, the exterior derivative is a linear operator, making the boundary constraint a linear differential constraint. More details on this can be found in the original paper.


## 3. Relaxing the Problem

So far, we’re minimizing $\| \delta_\Sigma \|_{\text{mass}}$ subject to $d\delta_\Sigma = \delta_\Gamma$. But we only allow $\delta_\Sigma$ that comes from actual surfaces. The obvious relaxation for the problem is optimizing over all possible 1-forms which satisfy the boundary constraint. As the authors detail, this relaxed version of the problem gives the same value as the non-relaxed version, but now we have much more flexibility in the search space. Formally, the final optimization problem is

$$
\boxed{
\min_{\eta \in \Omega^1(M), \ d\eta = \delta_\Gamma} \| \eta \|_{\text{mass}}
}
$$

This formulation of the problem is convex and has a linear constraint, making it easier to crack using standard non-linear optimization methods. In the original paper, this formulation is represented as problem ???.


## 4. Topological Considerations: Periodic Domains and Cohomology

To accelerate computation, we implement the solver to work in a periodic box $M = \mathbb{T}^3$ (i.e., a 3D torus). This allows use of Fast Fourier Transforms (FFT) to solve the necessary PDE constraints efficiently.

However, this introduces new challenges:
- Surfaces that wrap around the domain may satisfy $d\eta = \delta_\Gamma$, even if they don't produce a true surface in $\mathbb{R}^3$
- The solution space includes multiple cohomology classes, each representing a different homology type of surface, i.e. surfaces might have different number of holes.

To eliminate these ambiguities, we fix the cohomology class by computing the projected area vector:
$$
A = \int_\Sigma N_\Sigma \, dS = \frac{1}{2} \oint_\Gamma \gamma \times d\gamma
$$

and enforcing these additional constraints, 

$$
\int_M \vartheta_i \wedge \star \eta = A_i,i = 1,2,3
$$

where $\vartheta_i = dx_i$, and each $A_i$ corresponds to the signed projected area onto the $yz$, $zx$, and $xy$ planes respectively. These constraints ensure that the solution corresponds to a surface that can exist in $\mathbb{R}^3$ rather than wrapping around $\mathbb{T}^3$. See **Section 2.6** and **Appendix B** of the original paper for full details on how these constraints reduce the admissible set to embedded surfaces. 

After including these cohomology constraints, the optimization problem becomes this

$$
\min_{\eta \in \Omega^1(M), \ d\eta = \delta_\Gamma, A_i = \int_M dx_i \wedge \star \eta} \| \eta \|_{\text{mass}}
$$

## Decomposition

As the original authors describe, we can further simplify the optimization problem by using the Helmoltz-Hodge decomposition on $\eta$. We can write the full 1-form $\eta$ as a sum of an initial guess term $\eta_0$ and some form which is the result of applying exterior derivative to a smooth scalar function. In other words, we can write $\eta = \eta_0 + d\phi$, where $\phi \in C^\infty(M)$. The initial guess $\eta_0 \in \Omega^1(M)$ is a 1-form and also must satisfy both the boundary constraint: $d\eta_0 = \delta_\Gamma$ and the cohomology constraints: $\int_M dx_i \wedge \star \eta_0 = A_i$ for $i=1,2,3$
 
We can now write the final formulation of the problem, which is numbered as problem 6 in the original paper,

$$
\boxed{
\begin{aligned}
&\min_{\phi \in C^\infty(M)} \| \eta_0 + d\phi \|_{\text{mass}} \\
&\text{where } \eta_0 \in \Omega^1(M) \text{ satisfies:} \\
&\quad d\eta_0 = \delta_\Gamma \\
&\quad \int_M dx_i \wedge \star \eta_0 = A_i, \quad i = 1,2,3
\end{aligned}
}
$$

## Summary

We’ve transformed the Plateau problem from a nonlinear, hard-to-solve geometric variational problem into a problem which

- is a convex optimization over 1-forms
- has linear PDE constraints
- is efficiently solvable using Fast Fourier Transform in the periodic domain $\mathbf{T}^3$

This formulation is the foundation for computing minimal surfaces without meshes, without worrying about topology, and with guaranteed convergence.


# Coming up

In the next post, we’ll look at how to implement the required mathematical tools in Python using numpy. After we have successfully built the implementations of the different tools, we combine the theoretical groundwork we made in this post and the actual implementations to solve the problem numerically. We can then see that the approach created by the original authors truly works in this alternate techical landscape.

# Implementation: Part I - Initial guess $\eta_0$

In previous parts we have introduced the original problem and through various modifications and mathematical tools reduced the problem to a nice, hopefully solvable form. In this part we will go through finding a suitable initial guess $\eta_0$, which is used in the next part to find the optimal solution $\eta$.

As a reminder, we want to find an initial guess $\eta_0$, which satisfies the boundary constraint $d\eta_0 = \delta_\Gamma$ and the cohomology constraints $\int_M dx_i \wedge \star \eta_0 = A_i, i = 1, 2, 3$.

## How to find the initial guess?

As the original authors describe, the initial guess can be found by first creating the 2-form $\delta_\Gamma$ as a discrete field on $M = \mathbb{T}^3$.  
From that 2-form, one can find the 1-form with $d\eta_0 = \delta_\Gamma$ by solving the Biot-Savart equations for an auxiliary 2-form $\mathbf{\psi}$.

Adding the cohomology constraints is then just a simple addition to the 1-form $\eta_0$ by computing the area vector $\mathbf{A} = \oint \gamma \times d\gamma$ and adding the global shift to $\eta_0$.

Finally, the 1-form $\eta_0$ is discretized as a vector field $X_0$.

### How the discretization is set up

Everything is collocated at grid vertices and stored as a pointwise *density*.
The exterior derivative is the **forward** difference,

$$(d\phi)_i(v) = \frac{\phi(v + h e_i) - \phi(v)}{h},
\qquad \widehat{d}_i = \frac{e^{i k_i h} - 1}{h},$$

and the codifferential is its exact adjoint. This choice matters more than it
looks. Summing $|\widehat{d}_i|^2$ gives $4\sum_i \sin^2(k_i h/2)/h^2$, which is
exactly the 7-point stencil of the paper's Algorithm 3 — so the Poisson solver
really does invert $d^\top d$, and the ADMM $\phi$-step is an exact projection.

The paper instead defines $D$ as the midpoint rule (their eq. 23) while using the
7-point stencil to invert it. Those are different operators, and composing them
leaves a 71% relative residual in a step that is supposed to be an exact argmin.

Forward differences also *localize*: the gradient of a step occupies a single
cell. Since the minimizer is a Dirac-δ form concentrated on a surface, and the
mass norm sums $|X|$ without letting oscillations cancel, a delocalized
derivative inflates the objective — a spectral $d$ rings across the whole domain
and converges to a mass about 62% above the true area.

In [ ]:
# The discrete exterior calculus is exact, not merely consistent.
rng = np.random.default_rng(0)
phi = rng.standard_normal(grid.res)
eta = rng.standard_normal((*grid.res, 3))

print(f"d(d(phi))                        = {np.abs(spectral.d1(grid, spectral.d0(grid, phi))).max():.2e}")
print(f"<d0 phi, eta> - <phi, delta1 eta> = "
      f"{float((spectral.d0(grid, phi)*eta).sum()) - float((phi*spectral.delta1(grid, eta)).sum()):.2e}")
lap = spectral.delta1(grid, spectral.d0(grid, phi))
print(f"delta1(d0(phi)) vs Laplacian     = {np.abs(lap - spectral.laplace_psd(grid, phi)).max()/np.abs(lap).max():.2e}")

seven_point = sum(4/grid.h**2 * np.sin(k*grid.h/2)**2 for k in grid.k_space)
print(f"d^T d vs the paper's Algorithm 3 = {np.abs(grid.laplace_symbol - seven_point).max()/seven_point.max():.2e}")

The initial guess $\tilde{\eta}_0$ corresponds to a nice Biot-Savart field, familiar as the electromagnetic field for a path of electrical current.

## Area correction 
Below is the cohomology corrected field $\eta_0$.   
As long as the input curve $\Gamma$ does not go through the boundaries of the unit box, the surface represented by $\eta_0$ should be able to be embedded in $\mathbb{R}^3$, not just in $\mathbb{T}^3$. As described in the earlier posts and in the original paper, this eliminates the artifacts created by solving the differential equations on a 3-torus, a requirement of using FFT.

# Optimization


After finding the initial guess, we can optimize the surface to have the minimal area, while maintaining the boundary constraint.

The admissible set is $\eta_0 + \mathrm{im}(d)$, so the constrained problem
becomes unconstrained in a scalar potential $\phi$:

$$\min_{\phi}\ \|\eta_0 + d\phi\|_{\text{mass}}.$$

ADMM splits this into a Poisson solve for $\phi$ and a pointwise shrinkage for
$X$, coupled by a dual variable $\lambda$. Two details the paper gets wrong here:
its eq. (29) uses $\tau\lambda$ where completing the square on its own eq. (28)
gives $\lambda/\tau$ (the two agree only at $\tau = 1$, the default, which hides
the error), and it gives no guidance on choosing $\tau$ at all — even though the
shrinkage threshold $1/\tau$ is in absolute field units, so at $\tau=1$ a typical
$\eta_0$ is thresholded away entirely on the first iteration.

In [ ]:
# rtol=1e-3 is plenty for an area estimate; see the note below on why the mass
# settles long before the residuals do.
solution = solve_plateau(gamma, resolution=48, rtol=1e-3, max_iter=800)
solution

In [ ]:
exact = np.pi * radius**2
h = solution.history

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(h["mass"])
axes[0].axhline(exact, color="k", ls="--", label=rf"exact $\pi r^2$ = {exact:.4f}")
axes[0].set_xlabel("ADMM iteration"); axes[0].set_ylabel(r"$\|\eta\|_{mass}$")
axes[0].set_title("Mass converging to the disc area"); axes[0].legend()

axes[1].semilogy(h["primal_relative"], label="primal feasibility")
axes[1].semilogy(h["dual_relative"], label="dual feasibility")
axes[1].semilogy(h["criterion"], label="paper's $c$")
axes[1].set_xlabel("ADMM iteration"); axes[1].set_title("Residuals (relative)"); axes[1].legend()
plt.tight_layout()

print(f"mass = {solution.mass:.5f}   exact = {exact:.5f}   error = {(solution.mass-exact)/exact:+.2%}")
print(f"converged={solution.converged} after {solution.iterations} iterations")

The mass starts *below* the true area and rises. That is not a bug: λ begins at
zero, so the first shrinkage sees no dual correction and thresholds most of the
field away. It recovers over the following iterations.

Note also the paper's criterion `c` in the right-hand plot. It measures only how
far the iterates moved, so it falls whether the method converges or merely
stalls. The solver stops instead on the two KKT conditions — primal feasibility
$D\phi - X + X_0 \to 0$ and dual feasibility $D^\top\lambda \to 0$ — which is
what actually certifies optimality.

The residuals decay roughly like $1/k$, but notice how much earlier the mass
settles: it is worth four digits long before the residuals reach `rtol`.

## Recovering the surface

The optimizer $\eta = \delta_\Sigma$ is an impulse on the surface, not a mesh. To
render it we find the 0-form $u$ whose differential best matches $\eta$,

$$u = \arg\min_u \|du - \delta_\Sigma\|_{L^2},$$

which is the same normal-equation solve as the $\phi$-step. Theory says $u$ then
has a jump of exactly 1 across $\Sigma$, so an isosurface taken inside that jump
contains it. An isosurface is always closed, so the part extending past $\Gamma$
is clipped using $|\eta|$, which vanishes off the surface.

In [ ]:
vertices, faces = extract.extract_surface(solution)
print(f"{len(vertices)} vertices, {len(faces)} triangles")
print(f"mesh area   = {extract.surface_area(vertices, faces):.5f}")
print(f"mass norm   = {solution.mass:.5f}")
print(f"exact       = {exact:.5f}")

fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(projection="3d")
ax.plot_trisurf(vertices[:, 0], vertices[:, 1], faces, vertices[:, 2],
                cmap="viridis", alpha=0.9, linewidth=0)
t = np.linspace(0, 1, 200)
c = np.array([gamma(ti) for ti in t])
ax.plot(c[:, 0], c[:, 1], c[:, 2], "r-", lw=2.5, label=r"$\Gamma$")
ax.set_box_aspect([1, 1, 0.6]); ax.legend(); ax.set_title("Minimal surface spanning a circle")

## Validation

The circle is the only case here with a closed-form answer, so it is the one that
can be checked against a number rather than against itself. The error is first
order in *h*, and its size has a clean interpretation: the discrete surface is
thickened by about half a cell at its rim.

In [ ]:
rows = []
for N in (16, 24, 32, 48):
    s = solve_plateau(gamma, resolution=N, max_iter=300, sigma_cells=0.0)
    excess = s.mass - exact
    rows.append((N, 1/N, s.mass, excess/exact, excess/(2*np.pi*radius), 0.5/N))

print(f"{'N':>4} {'h':>8} {'mass':>9} {'rel err':>9} {'excess/perim':>13} {'h/2':>8}")
for N, hh, m, rel, per, half in rows:
    print(f"{N:4d} {hh:8.4f} {m:9.5f} {rel:+9.2%} {per:13.5f} {half:8.5f}")

Every *planar* curve gives another closed-form check, since its minimal surface
is just the region it bounds. The last column below is the same half-cell
boundary layer measured on three different shapes — the triangle's larger
relative error is entirely its higher perimeter-to-area ratio, not a different
failure mode, so corners are handled fine.

In [ ]:
tri = np.array([(0.3, 0.3, 0.5), (0.7, 0.35, 0.5), (0.5, 0.7, 0.5)])
cases = [
    ("circle r=0.3",   gamma,                                        np.pi*radius**2,  2*np.pi*radius),
    ("ellipse .35x.2", lambda t: curves.ellipse(t, a=0.35, b=0.2),   np.pi*0.35*0.2,   1.760),
    ("triangle",       curves.triangle,                              0.075,            1.2534),
]

print(f"{'curve':>16} {'N':>4} {'mass':>9} {'exact':>9} {'rel err':>9} {'excess/perim/(h/2)':>20}")
for name, g, ex, perim in cases:
    for N in (32, 48):
        s = solve_plateau(g, resolution=N, max_iter=500)
        print(f"{name:>16} {N:4d} {s.mass:9.5f} {ex:9.5f} {(s.mass-ex)/ex:+9.2%}"
              f" {(s.mass-ex)/perim/(0.5/N):20.2f}")

In [ ]:
# Other boundary curves. The trefoil's minimal surface is a genuine Seifert-like
# spanning surface, not anything disc-shaped.
fig = plt.figure(figsize=(13, 5))
for i, (name, g, area) in enumerate([
    ("trefoil", curves.trefoil, None),
    ("triangle", curves.triangle, None),
]):
    sol = solve_plateau(g, resolution=48, max_iter=250, area=area)
    v, f = extract.extract_surface(sol)
    ax = fig.add_subplot(1, 2, i+1, projection="3d")
    ax.plot_trisurf(v[:, 0], v[:, 1], f, v[:, 2], cmap="viridis", alpha=0.9, linewidth=0)
    t = np.linspace(0, 1, 400); c = np.array([g(ti) for ti in t])
    ax.plot(c[:, 0], c[:, 1], c[:, 2], "r-", lw=2)
    ax.set_title(f"{name}  (mass = {sol.mass:.4f})")
plt.tight_layout()

## Corrections to the published algorithm

Working through the paper turned up several details that are either wrong or
unstated. Each is fixed in `src/` with a test guarding it; `README.md` has the
full derivations.

1. **The φ-step's Laplacian does not match its own `D`** (their eq. 23 vs
   Algorithm 3) — a 71% relative residual in an exact argmin. Fixed by taking `d`
   to be the forward difference, which reproduces their 7-point stencil exactly.
2. **eq. (29) uses `τλ` where it needs `λ/τ`** — masked at the default τ = 1.
3. **Algorithm 5 line 21 contradicts eq. (23)** — the sharp operator is written as
   a difference where its own definition is an average.
4. **The Biot–Savart solve needs the positive semi-definite Laplacian** — the
   other sign yields `d(η₀) = −δ_Γ`, the reversed orientation.
5. **Algorithm 5 lines 12–18 drop a normalization** — no division by `|V|`.
6. **No guidance on τ** — and it cannot be scale-free.
7. **A convergence criterion that cannot distinguish converged from stalled.**

# Recap

In this blog post we went through a lot of the underlying mathematical tools we use in order to solve the Plateau problem.  
We discovered how from vector fields we can arrive at more intrinsic differential geometric objects and how to represent different geometric objects using more abstract objects.

In the next part we will get our hands dirty and start thinking about the actual optimization problem and then move on to actual implementation in the following posts.